In [ ]:
#| default_exp md

# Markdown

> For rich markdown and text rendering 

In [ ]:
#| export
from fasthtml.common import *
from basecampui.utils import *
from basecampui.common import *
from basecampui.interactive import *
from fasthtml.jupyter import *
import re
from mistlefoot import ExtendedHtmlRenderer, parse_attrs
from mistletoe import markdown

In [ ]:
app = FastHTML()
rt = app.route

In [ ]:
srv = JupyUvi(app, port=8003)

In [ ]:
#| export
class BasecampHtmlRenderer(ExtendedHtmlRenderer):
    def render_heading(self, token):
        _heading_classes = {
            1: "text-4xl font-bold tracking-tight",
            2: "text-3xl font-semibold tracking-tight",
            3: "text-2xl font-semibold tracking-tight",
            4: "text-xl font-semibold tracking-tight",
            5: "text-lg font-medium",
            6: "text-base font-medium",
        }

        inner = self.render_inner(token)
        attr_match = re.search(r'\s*\{([^}]+)\}$', inner)
        extra_attrs = ''
        if attr_match:
            extra_attrs = parse_attrs(attr_match.group(0))
            inner = inner[:attr_match.start()]
        cls = _heading_classes.get(token.level, '')
        return f'<h{token.level} class="{cls}"{extra_attrs}>{inner}</h{token.level}>'

    def render_list(self, token):
        if token.start is not None:
            tag = 'ol'
            lst = 'decimal'
            attr = f' start="{token.start}"' if token.start != 1 else ''
        else:
            tag = 'ul'
            lst = 'disc'
            attr = ''
        self._suppress_ptag_stack.append(not token.loose)
        inner = '\n'.join([self.render(child) for child in token.children])
        self._suppress_ptag_stack.pop()
        return f'<{tag} style="list-style-type: {lst}; padding-left: 1.5rem; display: flex; flex-direction: column; gap: 0.1rem;"{attr}>\n{inner}\n</{tag}>'
        
    def render_link(self, token) -> str:
        template = '<a href="{target}"{title} class="text-[rgb(135,169,255)] font-medium">{inner}</a>'
        target = self.escape_url(token.target)
        if token.title:
            title = ' title="{}"'.format(html.escape(token.title))
        else:
            title = ''
        inner = self.render_inner(token)
        return template.format(target=target, title=title, inner=inner)

    def render_auto_link(self, token):
        target = self.escape_url(token.target)
        return f'<a href="{target}" class="text-[rgb(135,169,255)] font-medium">{target}</a>'



In [ ]:
ExtendedHtmlRenderer??


```python
class ExtendedHtmlRenderer(HtmlRenderer):
    def __init__(self, *args, **kw): 
        super().__init__(FencedDiv, AttrLink, Subscript, Superscript, Highlight, Emoji, FootnoteRef,
            FootnoteEntry, Strikethrough, AutoLink, *args, **kw)
        self.footnotes = {}
        ListItem.pattern = re.compile(r'( {0,3})(\d{1,9}[.)]|[+\-*])($|\s+)')
        List.pattern = re.compile(r' {0,3}(?:\d{1,9}[.)]|[+\-*])(?:[ \t]*$|[ \t]+)')
    def render_subscript(self, token): return f'<sub>{token.content}</sub>'
    def render_superscript(self, token): return f'<sup>{token.content}</sup>'
    def render_highlight(self, token): return f'<mark>{self.render_inner(token)}</mark>'
    def render_strikethrough(self, token): return f'<del>{self.render_inner(token)}</del>'
    def render_emoji(self, token): return emoji_map.get(f':{token.name}:', f':{token.name}:')
    def render_auto_link(self, token): return f'<a href="{token.target}">{token.target}</a>'
    def render_footnote_ref(self, token): 
        lbl = token.label
        return f'<sup><a href="#fn-{lbl}" id="fnref-{lbl}">[{lbl}]</a></sup>'
    def render_footnote_entry(self, token):
        lbl = token.label
        return f'<div id="fn-{lbl}" class="footnote"><sup>{lbl}</sup> {token.content} <a href="#fnref-{lbl}">↩</a></div>\n'
    def render_list_item(self, token):
        inner = self.render_inner(token)
        match = re.match(r'\[( |x|X)\] ', inner)
        if not match: return f'<li>{inner}</li>\n'
        checked = 'checked' if match.group(1).lower() == 'x' else ''
        checkbox = f'<input type="checkbox" disabled {checked}>'
        content = inner[match.end():]
        return f'<li>{checkbox} {content}</li>\n'
    def render_attr_link(self, token):
        attrs = parse_attrs('{' + token.attr_str + '}')
        return f'<a href="{token.target}"{attrs}>{token.text}</a>'
    def render_fenced_div(self, token):
        attrs = parse_attrs(token.attr_str) if token.attr_str else ''
        return f'<div{attrs}>\n{self.render_inner(token)}</div>\n'

    def __enter__(self):
        _render_lock.acquire()
        try: return super().__enter__()
        except:
            _render_lock.release()
            raise

    def __exit__(self, *exc):
        try: return super().__exit__(*exc)
        finally:
            reset_tokens()
            _render_lock.release()
```

**File:** `/usr/local/lib/python3.12/site-packages/mistlefoot/core.py`

In [ ]:
from mistletoe import HtmlRenderer
HtmlRenderer??


```python
class HtmlRenderer(BaseRenderer):
    """
    HTML renderer class.

    See mistletoe.base_renderer module for more info.
    """
    def __init__(
        self,
        *extras,
        html_escape_double_quotes=False,
        html_escape_single_quotes=False,
        process_html_tokens=True,
        **kwargs
    ):
        """
        Args:
            extras (list): allows subclasses to add even more custom tokens.
            html_escape_double_quotes (bool): whether to also escape double
                quotes when HTML-escaping rendered text.
            html_escape_single_quotes (bool): whether to also escape single
                quotes when HTML-escaping rendered text.
            process_html_tokens (bool): whether to include HTML tokens in the
                processing. If `False`, HTML markup will be treated as plain
                text: e.g. input ``<br>`` will be rendered as ``&lt;br&gt;``.
            **kwargs: additional parameters to be passed to the ancestor's
                constructor.
        """
        self._suppress_ptag_stack = [False]
        final_extras = chain((HtmlBlock, HtmlSpan) if process_html_tokens else (), extras)
        super().__init__(*final_extras, **kwargs)
        self.html_escape_double_quotes = html_escape_double_quotes
        self.html_escape_single_quotes = html_escape_single_quotes

    def __exit__(self, *args):
        super().__exit__(*args)

    def render_to_plain(self, token) -> str:
        if token.children is not None:
            inner = [self.render_to_plain(child) for child in token.children]
            return ''.join(inner)
        return html.escape(token.content)

    def render_strong(self, token: span_token.Strong) -> str:
        template = '<strong>{}</strong>'
        return template.format(self.render_inner(token))

    def render_emphasis(self, token: span_token.Emphasis) -> str:
        template = '<em>{}</em>'
        return template.format(self.render_inner(token))

    def render_inline_code(self, token: span_token.InlineCode) -> str:
        template = '<code>{}</code>'
        inner = self.escape_html_text(token.children[0].content)
        return template.format(inner)

    def render_strikethrough(self, token: span_token.Strikethrough) -> str:
        template = '<del>{}</del>'
        return template.format(self.render_inner(token))

    def render_image(self, token: span_token.Image) -> str:
        template = '<img src="{}" alt="{}"{} />'
        if token.title:
            title = ' title="{}"'.format(html.escape(token.title))
        else:
            title = ''
        return template.format(token.src, self.render_to_plain(token), title)

    def render_link(self, token: span_token.Link) -> str:
        template = '<a href="{target}"{title}>{inner}</a>'
        target = self.escape_url(token.target)
        if token.title:
            title = ' title="{}"'.format(html.escape(token.title))
        else:
            title = ''
        inner = self.render_inner(token)
        return template.format(target=target, title=title, inner=inner)

    def render_auto_link(self, token: span_token.AutoLink) -> str:
        template = '<a href="{target}">{inner}</a>'
        if token.mailto:
            target = 'mailto:{}'.format(token.target)
        else:
            target = self.escape_url(token.target)
        inner = self.render_inner(token)
        return template.format(target=target, inner=inner)

    def render_escape_sequence(self, token: span_token.EscapeSequence) -> str:
        return self.render_inner(token)

    def render_raw_text(self, token: span_token.RawText) -> str:
        return self.escape_html_text(token.content)

    @staticmethod
    def render_html_span(token: span_token.HtmlSpan) -> str:
        return token.content

    def render_heading(self, token: block_token.Heading) -> str:
        template = '<h{level}>{inner}</h{level}>'
        inner = self.render_inner(token)
        return template.format(level=token.level, inner=inner)

    def render_quote(self, token: block_token.Quote) -> str:
        elements = ['<blockquote>']
        self._suppress_ptag_stack.append(False)
        elements.extend([self.render(child) for child in token.children])
        self._suppress_ptag_stack.pop()
        elements.append('</blockquote>')
        return '\n'.join(elements)

    def render_paragraph(self, token: block_token.Paragraph) -> str:
        if self._suppress_ptag_stack[-1]:
            return '{}'.format(self.render_inner(token))
        return '<p>{}</p>'.format(self.render_inner(token))

    def render_block_code(self, token: block_token.BlockCode) -> str:
        template = '<pre><code{attr}>{inner}</code></pre>'
        if token.language:
            attr = ' class="{}"'.format('language-{}'.format(html.escape(token.language)))
        else:
            attr = ''
        inner = self.escape_html_text(token.content)
        return template.format(attr=attr, inner=inner)

    def render_list(self, token: block_token.List) -> str:
        template = '<{tag}{attr}>\n{inner}\n</{tag}>'
        if token.start is not None:
            tag = 'ol'
            attr = ' start="{}"'.format(token.start) if token.start != 1 else ''
        else:
            tag = 'ul'
            attr = ''
        self._suppress_ptag_stack.append(not token.loose)
        inner = '\n'.join([self.render(child) for child in token.children])
        self._suppress_ptag_stack.pop()
        return template.format(tag=tag, attr=attr, inner=inner)

    def render_list_item(self, token: block_token.ListItem) -> str:
        if len(token.children) == 0:
            return '<li></li>'
        inner = '\n'.join([self.render(child) for child in token.children])
        inner_template = '\n{}\n'
        if self._suppress_ptag_stack[-1]:
            if token.children[0].__class__.__name__ == 'Paragraph':
                inner_template = inner_template[1:]
            if token.children[-1].__class__.__name__ == 'Paragraph':
                inner_template = inner_template[:-1]
        return '<li>{}</li>'.format(inner_template.format(inner))

    def render_table(self, token: block_token.Table) -> str:
        # This is actually gross and I wonder if there's a better way to do it.
        #
        # The primary difficulty seems to be passing down alignment options to
        # reach individual cells.
        template = '<table>\n{inner}</table>'
        if hasattr(token, 'header'):
            head_template = '<thead>\n{inner}</thead>\n'
            head_inner = self.render_table_row(token.header, is_header=True)
            head_rendered = head_template.format(inner=head_inner)
        else:
            head_rendered = ''
        body_template = '<tbody>\n{inner}</tbody>\n'
        body_inner = self.render_inner(token)
        body_rendered = body_template.format(inner=body_inner)
        return template.format(inner=head_rendered + body_rendered)

    def render_table_row(self, token: block_token.TableRow, is_header=False) -> str:
        template = '<tr>\n{inner}</tr>\n'
        inner = ''.join([self.render_table_cell(child, is_header)
                         for child in token.children])
        return template.format(inner=inner)

    def render_table_cell(self, token: block_token.TableCell, in_header=False) -> str:
        template = '<{tag}{attr}>{inner}</{tag}>\n'
        tag = 'th' if in_header else 'td'
        if token.align is None:
            align = 'left'
        elif token.align == 0:
            align = 'center'
        elif token.align == 1:
            align = 'right'
        attr = ' align="{}"'.format(align)
        inner = self.render_inner(token)
        return template.format(tag=tag, attr=attr, inner=inner)

    @staticmethod
    def render_thematic_break(token: block_token.ThematicBreak) -> str:
        return '<hr />'

    @staticmethod
    def render_line_break(token: span_token.LineBreak) -> str:
        return '\n' if token.soft else '<br />\n'

    @staticmethod
    def render_html_block(token: block_token.HtmlBlock) -> str:
        return token.content

    def render_document(self, token: block_token.Document) -> str:
        self.footnotes.update(token.footnotes)
        inner = '\n'.join([self.render(child) for child in token.children])
        return '{}\n'.format(inner) if inner else ''

    def escape_html_text(self, s: str) -> str:
        """
        Like `html.escape()`, but this  looks into the current rendering options
        to decide which of the quotes (double, single, or both) to escape.

        Intended for escaping text content. To escape content of an attribute,
        simply call `html.escape()`.
        """
        s = s.replace("&", "&amp;")  # Must be done first!
        s = s.replace("<", "&lt;")
        s = s.replace(">", "&gt;")
        if self.html_escape_double_quotes:
            s = s.replace('"', "&quot;")
        if self.html_escape_single_quotes:
            s = s.replace('\'', "&#x27;")
        return s

    @staticmethod
    def escape_url(raw: str) -> str:
        """
        Escape urls to prevent code injection craziness. (Hopefully.)
        """
        return html.escape(quote(raw, safe='/#:()*?=%@+,&;'))
```

**File:** `/usr/local/lib/python3.12/site-packages/mistletoe/html_renderer.py`

In [ ]:
#| export
def render_md(md:str):
    return NotStr(markdown(md, BasecampHtmlRenderer))

In [ ]:
md = """
# This thing
There is a **lot** of content here. For example __consider__ that we could:
```python
def main(x):
    return x + 1
```

And if that doesn't work then:
- Try to do a differnet thing
- Try to be better
- Try something else

https://github.com/taya-nicholas/basecampui
"""

pw(
    Div(
        render_md(md)
    )
)

HTML(<iframe srcdoc="&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;title&gt;FastHTML page&lt;/title&gt;
    &lt;link rel=&quot;canonical&quot; href=&quot;https://testserver/_QQt-cNTwSHaH9QWuGXbI5Q&quot;&gt;
    &lt;meta charset=&quot;utf-8&quot;&gt;
    &lt;meta name=&quot;viewport&quot; content=&quot;width=device-width, initial-scale=1, viewport-fit=cover&quot;&gt;
&lt;script src=&quot;https://cdn.jsdelivr.net/npm/htmx.org@2.0.7/dist/htmx.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/answerdotai/fasthtml-js@1.0.12/fasthtml.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/answerdotai/surreal@main/surreal.js&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/gh/gnat/css-scope-inline@main/script.js&quot;&gt;&lt;/script&gt;&lt;script&gt;
(() =&gt; {
  const stored = localStorage.getItem(&#x27;themeMode&#x27;);
  const dark = stored ? stored === &#x27;dark&#x27; : matchMedia(&#x27;(prefers-color-scheme: dark)&#x27;).matches;
  if (dark) document.documentElement.classList.add(&#x27;dark&#x27;);
  
  document.addEventListener(&#x27;basecoat:theme&#x27;, (e) =&gt; {
    const mode = e.detail?.mode || (document.documentElement.classList.contains(&#x27;dark&#x27;) ? &#x27;light&#x27; : &#x27;dark&#x27;);
    const isDark = mode === &#x27;dark&#x27;;
    document.documentElement.classList.toggle(&#x27;dark&#x27;, isDark);
    localStorage.setItem(&#x27;themeMode&#x27;, isDark ? &#x27;dark&#x27; : &#x27;light&#x27;);
  });
})();
&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/npm/@tailwindcss/browser@4&quot;&gt;&lt;/script&gt;&lt;script src=&quot;https://cdn.jsdelivr.net/npm/basecoat-css@0.3.11/dist/js/all.min.js&quot;&gt;&lt;/script&gt;    &lt;link rel=&quot;stylesheet&quot; href=&quot;https://cdn.jsdelivr.net/npm/basecoat-css@0.3.11/dist/basecoat.cdn.min.css&quot;&gt;
    &lt;style type=&quot;text/tailwindcss&quot;&gt;
@theme inline {
  --color-background: var(--background);
  --color-foreground: var(--foreground);
  --color-card: var(--card);
  --color-card-foreground: var(--card-foreground);
  --color-popover: var(--popover);
  --color-popover-foreground: var(--popover-foreground);
  --color-primary: var(--primary);
  --color-primary-foreground: var(--primary-foreground);
  --color-secondary: var(--secondary);
  --color-secondary-foreground: var(--secondary-foreground);
  --color-muted: var(--muted);
  --color-muted-foreground: var(--muted-foreground);
  --color-accent: var(--accent);
  --color-accent-foreground: var(--accent-foreground);
  --color-destructive: var(--destructive);
  --color-destructive-foreground: var(--destructive-foreground);
  --color-border: var(--border);
  --color-input: var(--input);
  --color-ring: var(--ring);
  --color-chart-1: var(--chart-1);
  --color-chart-2: var(--chart-2);
  --color-chart-3: var(--chart-3);
  --color-chart-4: var(--chart-4);
  --color-chart-5: var(--chart-5);
  --color-sidebar: var(--sidebar);
  --color-sidebar-foreground: var(--sidebar-foreground);
  --color-sidebar-primary: var(--sidebar-primary);
  --color-sidebar-primary-foreground: var(--sidebar-primary-foreground);
  --color-sidebar-accent: var(--sidebar-accent);
  --color-sidebar-accent-foreground: var(--sidebar-accent-foreground);
  --color-sidebar-border: var(--sidebar-border);
  --color-sidebar-ring: var(--sidebar-ring);
  --radius: var(--radius);
}
&lt;/style&gt;
    &lt;style&gt;.lucide-icon { stroke: currentColor; fill: none; stroke-width: 2; stroke-linecap: round; stroke-linejoin: round; }&lt;/style&gt;
&lt;svg xmlns=&quot;http://www.w3.org/2000/svg&quot; style=&quot;display: none&quot;&gt;&lt;defs&gt;&lt;symbol id=&quot;lc-chevron-down&quot;&gt;&lt;path d=&quot;m6 9 6 6 6-6&quot;&gt;&lt;/path&gt;&lt;/symbol&gt;&lt;symbol id=&quot;lc-ellipsis&quot;&gt;&lt;circle cx=&quot;12&quot; cy=&quot;12&quot; r=&quot;1&quot;&gt;&lt;/circle&gt;&lt;circle cx=&quot;19&quot; cy=&quot;12&quot; r=&quot;1&quot;&gt;&lt;/circle&gt;&lt;circle c